#### For this pipeline I used .env for credentials. If you do not have a .env file please make one and put in your credentials.

In [3]:
import csv
import pymongo
import random
from datetime import datetime
import os
from dotenv import load_dotenv
import glob

load_dotenv()

MONGO_USER = os.getenv("MONGO_USER")
MONGO_PASS = os.getenv("MONGO_PASS")
MONGO_HOST = os.getenv("MONGO_HOST")

uri = f"mongodb+srv://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}/?retryWrites=true&w=majority"
client = pymongo.MongoClient(uri)

db = client["flight_delays"]
collection = db["flights"]

# --- Helper to safely convert to float ---
def to_float(val):
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

def to_int(val):
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return None

# Read ALL rows from all CSVs first
all_rows = []
csv_files = glob.glob("../data/*.csv")

for csv_file in csv_files:
    print(f"Reading {csv_file}...")
    with open(csv_file, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            all_rows.append(row)

print(f"Total rows across all files: {len(all_rows)}")

# Randomly sample 50,000 rows to get a spread across all dates
SAMPLE_SIZE = 50000
if len(all_rows) > SAMPLE_SIZE:
    all_rows = random.sample(all_rows, SAMPLE_SIZE)
    print(f"Sampled down to {SAMPLE_SIZE} rows")

# Now transform and insert
docs = []
count = 0

for row in all_rows:
    doc = {
        "flight_date": {
            "date": row.get("FL_DATE", "").strip(),
            "weekday": row.get("DAY_OF_WEEK").strip()
        },
        "carrier": {
                "code": row.get("OP_CARRIER", "").strip(),
                "airline_id": to_int(row.get("OP_CARRIER_AIRLINE_ID")),
                "unique_carrier": row.get("OP_UNIQUE_CARRIER", "").strip()
            },
            "flight_number": to_int(row.get("OP_CARRIER_FL_NUM")),
            "origin": {
                "airport_id": to_int(row.get("ORIGIN_AIRPORT_ID")),
                "city": row.get("ORIGIN_CITY_NAME", "").strip(),
                "state": row.get("ORIGIN_STATE_ABR", "").strip()
            },
            "destination": {
                "airport_id": to_int(row.get("DEST_AIRPORT_ID")),
                "city_market_id": to_int(row.get("DEST_CITY_MARKET_ID"))
            },
            "schedule": {
                "crs_dep_time": row.get("CRS_DEP_TIME", "").strip(),
                "crs_arr_time": row.get("CRS_ARR_TIME", "").strip(),
                "crs_elapsed_time": to_float(row.get("CRS_ELAPSED_TIME"))
            },
            "actual": {
                "dep_time": row.get("DEP_TIME", "").strip(),
                "arr_time": row.get("ARR_TIME", "").strip(),
                "actual_elapsed_time": to_float(row.get("ACTUAL_ELAPSED_TIME")),
                "air_time": to_float(row.get("AIR_TIME")),
                "taxi_out": to_float(row.get("TAXI_OUT")),
                "taxi_in": to_float(row.get("TAXI_IN")),
                "wheels_off": row.get("WHEELS_OFF", "").strip(),
                "wheels_on": row.get("WHEELS_ON", "").strip()
            },
            "delay": {
                "dep_delay": to_float(row.get("DEP_DELAY")),
                "dep_delay_new": to_float(row.get("DEP_DELAY_NEW")),
                "dep_del15": to_float(row.get("DEP_DEL15")),
                "arr_delay": to_float(row.get("ARR_DELAY")),
                "arr_delay_new": to_float(row.get("ARR_DELAY_NEW")),
                "is_delayed": to_float(row.get("ARR_DELAY", 0)) is not None and (to_float(row.get("ARR_DELAY", 0)) or 0) >= 15,
                "carrier_delay": to_float(row.get("CARRIER_DELAY")),
                "weather_delay": to_float(row.get("WEATHER_DELAY")),
                "nas_delay": to_float(row.get("NAS_DELAY")),
                "security_delay": to_float(row.get("SECURITY_DELAY")),
                "late_aircraft_delay": to_float(row.get("LATE_AIRCRAFT_DELAY"))
            },
            "cancelled": to_float(row.get("CANCELLED")) == 1.0,
            "cancellation_code": row.get("CANCELLATION_CODE", "").strip() or None,
            "diverted": to_float(row.get("DIVERTED")) == 1.0,
            "distance_group": to_int(row.get("DISTANCE_GROUP"))
    }
    docs.append(doc)
    count += 1

    if len(docs) == 5000:
        collection.insert_many(docs)
        print(f"Inserted {count} documents...")
        docs = []

if docs:
    collection.insert_many(docs)

print(f"Done! {count} documents inserted.")

Reading ../data/T_ONTIME_REPORTING_jan.csv...
Reading ../data/T_ONTIME_REPORTING_mar.csv...
Reading ../data/T_ONTIME_REPORTING_feb.csv...
Total rows across all files: 1645503
Sampled down to 50000 rows
Inserted 5000 documents...
Inserted 10000 documents...
Inserted 15000 documents...
Inserted 20000 documents...
Inserted 25000 documents...
Inserted 30000 documents...
Inserted 35000 documents...
Inserted 40000 documents...
Inserted 45000 documents...
Inserted 50000 documents...
Done! 50000 documents inserted.


In [6]:
# Run these in your Colab notebook after connecting to MongoDB

import pymongo
from getpass import getpass

MONGO_USER = "tabele17"
MONGO_PASS = getpass("Enter MongoDB password: ")
MONGO_HOST = "cluster0.kasq2.mongodb.net"

uri = f"mongodb+srv://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}/?retryWrites=true&w=majority"
client = pymongo.MongoClient(uri)
db = client["flight_delays"]
collection = db["flights"]

# Total documents
total = collection.count_documents({})
print(f"Total documents: {total}")

# Unique carriers
carriers = collection.distinct("carrier.code")
print(f"Unique carriers: {len(carriers)} — {sorted(carriers)}")

# Unique origin airports
origins = collection.distinct("origin.airport_id")
print(f"Unique origin airports: {len(origins)}")

# Unique destination airports
dests = collection.distinct("destination.airport_id")
print(f"Unique destination airports: {len(dests)}")

# Delayed flights (arr_delay >= 15)
delayed = collection.count_documents({"delay.is_delayed": True})
print(f"Delayed flights: {delayed} ({delayed/total*100:.1f}%)")

# Cancelled flights
cancelled = collection.count_documents({"cancelled": True})
print(f"Cancelled flights: {cancelled} ({cancelled/total*100:.1f}%)")

# Date range
dates = collection.distinct("flight_date.date")
print(f"Date range: {min(dates)} to {max(dates)}")

Total documents: 60000
Unique carriers: 14 — ['AA', 'AS', 'B6', 'DL', 'F9', 'G4', 'HA', 'MQ', 'NK', 'OH', 'OO', 'UA', 'WN', 'YX']
Unique origin airports: 329
Unique destination airports: 329
Delayed flights: 13289 (22.1%)
Cancelled flights: 1747 (2.9%)
Date range: 1/1/2025 12:00:00 AM to 3/9/2025 12:00:00 AM
